## Práctica, Aplicaciones con Hugging Face y Gradio

En el siguiente proyecto vamos a utilizar los modelos existentes en la libreria de Hugging Face para crear una aplicación para realizar tareas de ASR (reconocimiento automático del habla), también generaremos una interfaz de usuario mediante la libreria de Gradio.

En primer lugar cargamos nuestra API-Key de Hugging face para poder trabajar con sus modelos.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

print(f"Token loaded: {hf_token is not None}")

Token loaded: False


### Separación de las voces de la instrumental

Se realiza una separación de la pista sobre la que vamos a trabajar con el objeto de obtener mejores resultados en el reconocimiento automático del habla. Para ello hacemos uso del modelo open source Demucs y almacenamos los output de audio en ficheros utilizando la librería soundfile.

In [2]:
import sys
import os
from demucs import pretrained
from demucs.apply import apply_model
import torchaudio
import torch

def separate_audio(input_path: str = "audio\Duvet.mp3", output_dir: str = "stems"):
    """
    Separates an input song into stems using Demucs.
    Stems: vocals, drums, bass, other.
    """
    # Crea el directorio de pistas si no existe
    os.makedirs(output_dir, exist_ok=True)

    # Cargar modelo preentrenado Demucs
    model = pretrained.get_model('htdemucs') 
    # Seteo el modelo en modo evaluación (desactiva comportamientos de entrenamiento) 
    model.eval()

    # Cargar fichero de audio
    wav, sr = torchaudio.load(input_path)
    # Evito conversión a mono - mantengo canales originales para Demucs
    wav = wav.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
   
    model = model.to(wav.device)

    # Corro el modelo sobre el audio
    with torch.no_grad():
        estimates = apply_model(model, wav[None], split=True, overlap=0.25)[0]

    # Escribir las pistas en archivos separados
    for source, audio in zip(model.sources, estimates):
        output_path = os.path.join(output_dir, f"{source}.mp3")
        torchaudio.save(output_path, audio.cpu(), sample_rate=sr)
        print(f"Fichero guardado: {output_path}")

    print("\n Separación exitosa. Las pistas de audio se encuentran en:", os.path.abspath(output_dir))

# Execute the separation
separate_audio()

Fichero guardado: stems\drums.mp3
Fichero guardado: stems\bass.mp3
Fichero guardado: stems\other.mp3
Fichero guardado: stems\vocals.mp3

 Separación exitosa. Las pistas de audio se encuentran en: c:\Users\sabax\OneDrive\Desktop\Trabajo-IA-1\stems


#### Tarea de ASR con Whisper en local

Uso del modelo de la librería de Hugging Face en local a través de un pipeline para realizar tareas de ASR con canciones. Nuestro objetivo en este caso es obtener la una transcripción de la letra de la canción y almacenarla en un fichero de texto con el que trabajaremos posteriormente.

In [4]:
import os
import librosa
from transformers import pipeline
import torch

# --- Configuración ---
AUDIO_FILE_PATH = os.path.join("stems", "vocals.mp3")
TRANSCRIPTION_DIR = "transcriptions"
TRANSCRIPTION_FILE_PATH = os.path.join(TRANSCRIPTION_DIR, "transcription_with_timestamps_duvet.txt")

# --- 1. Verificar Configuración y Cargar Modelo ---
print("--- Script de Transcripción ASR ---")

# Comprobar si la GPU está disponible
if torch.cuda.is_available():
    print(f"GPU disponible. Usando dispositivo: {torch.cuda.get_device_name(0)}")
    device = 0
else:
    print("GPU no encontrada. Usando CPU en su lugar. Esto será lento.")
    device = -1

# Cargar el pipeline de ASR
print("\nCargando modelo Whisper... (Esto puede tardar un momento)")
try:
    pipe = pipeline(
        "automatic-speech-recognition", 
        model="openai/whisper-large-v3",
        device=device,
        dtype=torch.float16)
    print("Modelo cargado con éxito.")
except Exception as e:
    print(f"Error al cargar el modelo: {e}")
    # Salir de la celda si el modelo no se puede cargar
    exit()

# --- 2. Cargar y Transcribir Audio ---
print(f"\nProcesando archivo de audio: {AUDIO_FILE_PATH}")

# Comprobar si el archivo de audio existe
if not os.path.exists(AUDIO_FILE_PATH):
    print(f"Error: No se encontró el archivo de audio en '{AUDIO_FILE_PATH}'")
else:
    try:
        # Cargar audio usando librosa
        audio, sr = librosa.load(AUDIO_FILE_PATH, sr=16000)
        
        print("Transcribiendo audio... (Esto puede tardar dependiendo de la duración de la pista)")
        result = pipe(audio, return_timestamps=True, generate_kwargs={"task": "transcribe", "language": "en"})
        print("Transcripción completada.")

        # --- 3. Guardar y Mostrar Resultados ---
        
        # Asegurarse de que exista el directorio de salida 
        os.makedirs(TRANSCRIPTION_DIR, exist_ok=True)
        
        # Guardar la transcripción detallada con marcas de tiempo
        with open(TRANSCRIPTION_FILE_PATH, "w", encoding="utf-8") as f:
            f.write("--- Transcripción Completa ---\n")
            f.write(result['text'].strip() + "\n\n")
            f.write("--- Segmentos con Marcas de Tiempo ---\n")
            for chunk in result['chunks']:
                start_time = round(chunk['timestamp'][0], 2)
                end_time = round(chunk['timestamp'][1], 2)
                start_formatted = f"{int(start_time//60):02d}:{int(start_time%60):02d}"
                end_formatted = f"{int(end_time//60):02d}:{int(end_time%60):02d}"
                f.write(f"[{start_formatted} -> {end_formatted}] {chunk['text'].strip()}\n")
        
        print(f"\nTranscripción guardada en: {TRANSCRIPTION_FILE_PATH}")
        
        # Imprimir el texto final de la transcripción en la consola
        print("\n--- Resultado de la Transcripción ---")
        print(result['text'])

    except Exception as e:
        print(f"Ocurrió un error durante el procesamiento o la transcripción del audio: {e}")


--- Script de Transcripción ASR ---
GPU disponible. Usando dispositivo: NVIDIA GeForce RTX 4060 Laptop GPU

Cargando modelo Whisper... (Esto puede tardar un momento)


Device set to use cuda:0


Modelo cargado con éxito.

Procesando archivo de audio: stems\vocals.mp3
Transcribiendo audio... (Esto puede tardar dependiendo de la duración de la pista)
Transcripción completada.

Transcripción guardada en: transcriptions\transcription_with_timestamps_duvet.txt

--- Resultado de la Transcripción ---
 And you don't seem to understand A shame you seemed an honest man And all the fears you hold so dear Will turn to whisper in your ear And you know what they say might hurt you And you know that it means too much And you don't even feel a thing I am falling, I am fading I have lost it all And you don't seem the lying kind A shame that I can read your mind And all the things that I read there Candlelit smile, the weep of shame And you know I don't mean to hurt you But you know that it means so much And you don't even feel a thing I am falling, I am fading I am drowning, coming to breathe I am hurting, I have lost it all I am losing, coming to breathe Oh, oh, oh, oh, oh, oh, oh, oh, oh, oh

## Gradio

In [12]:
import gradio as gr
from PIL import Image
import re
import os

# 🔁 mm:ss → segundos
def to_seconds(ts):
    mins, secs = map(int, ts.split(":"))
    return mins * 60 + secs

# 📖 TXT → subtítulos JSON
def load_subtitles(filepath):
    if not os.path.exists(filepath):
        print(f"⚠️ Archivo de subtítulos no encontrado: {filepath}")
        return None

    pattern = r"\[(\d\d:\d\d) -> (\d\d:\d\d)\] (.*)"
    subs = []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            m = re.match(pattern, line.strip())
            if m:
                start = to_seconds(m.group(1))
                end = to_seconds(m.group(2))
                text = m.group(3)
                subs.append({"timestamp": [float(start), float(end)], "text": text})
    return subs if subs else None

# 🎵 Canciones
songs = {
    "Bring Me To Life — Evanescence": {
        "audio": "audio/bring-me-to-life-evanescence.mp3",
        "cover": "covers/bringmeto.jpg",
        "subtitles": "transcriptions/transcription_with_timestamps.txt"
    },
    "Duvet — Boa": {
        "audio": "audio/Duvet.mp3",
        "cover": "covers/duvet.jpg",
        "subtitles": "transcriptions/transcription_with_timestamps_duvet.txt"
    }
}

# 🔄 Actualizar reproductor
def update_player(song_name):
    song = songs[song_name]

    cover_img = Image.open(song["cover"])
    subs = load_subtitles(song["subtitles"])

    return (
        cover_img,
        gr.update(value=song["audio"], subtitles=subs, autoplay=False),
        f"<div class='songtitle'>{song_name}</div>"
    )

# 🎨 Estilo Spotify
css = """
body { background-color: #121212; font-family: 'Helvetica', sans-serif; }

#title {
    color:white; font-size:32px; font-weight:bold;
    text-align:center; margin-bottom:20px;
}

.card {
    background: rgba(24,24,24,0.92);
    border: 1px solid rgba(255,255,255,0.08);
    border-radius:25px; padding:28px;
    max-width:450px; margin:auto; text-align:center;
    box-shadow: 0 8px 18px rgba(0,0,0,0.7);
    backdrop-filter: blur(10px);
}

.songtitle {
    color:#1DB954; font-size:22px; font-weight:bold;
    margin-top:14px; margin-bottom:8px;
}

img { border-radius: 15px; transition: .3s; }
img:hover { transform: scale(1.03); box-shadow: 0 6px 18px rgba(0,0,0,0.6); }
"""

# 🚀 UI
with gr.Blocks(css=css) as demo:
    gr.HTML("<div id='title'>🔥 Música To Guapa</div>")

    with gr.Column(elem_classes="card"):

        song_dropdown = gr.Dropdown(
            choices=list(songs.keys()),
            value=list(songs.keys())[0],
            label="Selecciona una canción"
        )

        cover_output = gr.Image(show_label=False)
        title_output = gr.HTML()
        audio_output = gr.Audio(show_label=False)

        song_dropdown.change(
            fn=update_player,
            inputs=song_dropdown,
            outputs=[cover_output, audio_output, title_output]
        )

        # Inicializar
        init = list(songs.keys())[0]
        cover_val, audio_val, title_val = update_player(init)
        cover_output.value = cover_val
        audio_output.value = songs[init]["audio"]
        audio_output.subtitles = load_subtitles(songs[init]["subtitles"])
        title_output.value = title_val

demo.launch()


* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
* To create a public link, set `share=True` in `launch()`.
